In [17]:
import torch
import os
%set_env TOKENIZERS_PARALLELISM=false
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

env: TOKENIZERS_PARALLELISM=false
Using device: cuda


In [18]:
import torch
def test1(device, d):
    w_qkv = torch.randn(size=(d, 3*d)).to(device)  # dims : d x 3d (qkv  matrix)
    x = torch.randn(size=(10,d)).to(device)    # 10xd (residual stram of 10 positions - 10 rows)
    w_q = w_qkv[:, 0:d].clone().contiguous()    # dxd (taking only the first d columns is like taking w_q)
    res1= x@w_qkv #(10, 3d)
    res2= x @ w_q # (10, d)
    q_from_res_1 = res1[:, 0:d] # (10, d)
    print(torch.max(q_from_res_1-res2))
    assert torch.equal(q_from_res_1, res2)  
test1("cpu", 512)

tensor(0.)


In [19]:
import pytest
import torch
from jaxtyping import Float
from torch.testing import assert_close
import torch.nn as nn
from transformer_lens.components import Attention
from transformer_lens.components import LayerNorm
from transformer_lens.components import HookedESM3MLP, swiglu_correction_fn
from transformer_lens.components import HookedEsm3UnifiedTransformerBlock
from esm.layers.attention import MultiHeadAttention
from esm.layers.blocks import swiglu_ln_ffn, UnifiedTransformerBlock
from transformer_lens.HookedTransformerConfig import HookedTransformerConfig
import functools
import einops
from esm.utils.constants.esm3 import data_root
import math
from transformer_lens import HookedESM3,SupportedESM3Config
from esm.pretrained import (
    ESM3_sm_open_v0,
)
from esm.models.esm3 import ESM3
import random
import torch.nn.functional as F
from esm.tokenization import get_esm3_model_tokenizers
from esm.utils.structure.protein_chain import ProteinChain


In [20]:
config = SupportedESM3Config(
    use_attn_result=True,
    use_split_qkv_input=True,
    use_hook_mlp_in=True,
    use_attn_in=True,
    esm3_output_type="all",
    esm3_use_torch_layer_norm=True,
    esm3_use_torch_attention_calc=True,
    esm3_use_org_rotary=True
)
esm3_hooked1 = HookedESM3.from_pretrained(esm_cfg=config, device=device)
esm3_original1 = ESM3_sm_open_v0(device).to(device)


If using ESM3 for interpretability research, keep in mind that ESM3 has some significant architectural differences to Language transformers like GPT.


Moving model to device:  cuda
Loaded pretrained model esm3_sm_open_v1 into HookedESM3


In [21]:
tokenizers = get_esm3_model_tokenizers()
esm3_original1.eval()  # Switch to evaluation mode to save memory
esm3_hooked1.eval()
sequence = "MGREFGNLTRMRHVISYSLSPFEQRAYPHVFTKGIPNVLRRIRESFFRVVPQFVVFYLIYTWGTEEFERSKRKNPAAYENDK"
tokens = tokenizers.sequence(sequence, return_tensors="pt", add_special_tokens=True,padding=True)['input_ids'].to(device)

In [22]:
with torch.no_grad():
    output1 = esm3_original1.forward(
        sequence_tokens=tokens
    )
with torch.no_grad():
    output2 = esm3_hooked1.forward(
    sequence_tokens=tokens
)

In [23]:
print(torch.max(torch.abs(output1.sequence_logits-output2.sequence_logits)))
print(torch.max(torch.abs(output1.structure_logits-output2.structure_logits)))
print(torch.max(torch.abs(output1.function_logits-output2.function_logits)))
print(torch.max(torch.abs(output1.residue_logits-output2.residue_logits)))
print(torch.max(torch.abs(output1.secondary_structure_logits-output2.secondary_structure_logits)))
print(torch.max(torch.abs(output1.sasa_logits-output2.sasa_logits)))

tensor(7.6294e-06, device='cuda:0')
tensor(4.7684e-05, device='cuda:0')
tensor(6.1035e-05, device='cuda:0')
tensor(3.9339e-05, device='cuda:0')
tensor(7.6294e-06, device='cuda:0')
tensor(7.6294e-06, device='cuda:0')


In [28]:
# Define dictionaries to store activations
activations_inputs = {}
activations_outputs = {}

# Define the hook function
def save_activation(name):
    def hook(module, input, output):
        # Save inputs
        if isinstance(input, tuple):  # Handle tuple inputs
            activations_inputs[name] = [i.detach() if isinstance(i, torch.Tensor) else i for i in input]
        elif isinstance(input, torch.Tensor):
            activations_inputs[name] = input.detach()
        else:
            activations_inputs[name] = input  # Handle non-tensor inputs

        # Save outputs
        if isinstance(output, tuple):  # Handle tuple outputs
            activations_outputs[name] = [o.detach() if isinstance(o, torch.Tensor) else o for o in output]
        elif isinstance(output, torch.Tensor):
            activations_outputs[name] = output.detach()
        else:
            activations_outputs[name] = output  # Handle non-tensor outputs
    return hook

# Register hooks for all modules
for name, module in esm3_original1.named_modules():
    module.register_forward_hook(save_activation(name))



In [29]:
with torch.no_grad():
    output1 = esm3_original1.forward(
        sequence_tokens=tokens
    )

In [30]:
activations_outputs.keys()

dict_keys(['encoder.sequence_embed', 'encoder.plddt_projection', 'encoder.structure_per_res_plddt_projection', 'encoder.structure_tokens_embed', 'encoder.ss8_embed', 'encoder.sasa_embed', 'encoder.function_embed.0', 'encoder.function_embed.1', 'encoder.function_embed.2', 'encoder.function_embed.3', 'encoder.function_embed.4', 'encoder.function_embed.5', 'encoder.function_embed.6', 'encoder.function_embed.7', 'encoder.residue_embed', 'encoder', 'transformer.blocks.0.attn.layernorm_qkv.0', 'transformer.blocks.0.attn.layernorm_qkv.1', 'transformer.blocks.0.attn.layernorm_qkv', 'transformer.blocks.0.attn.q_ln', 'transformer.blocks.0.attn.k_ln', 'transformer.blocks.0.attn.rotary', 'transformer.blocks.0.attn.out_proj', 'transformer.blocks.0.attn', 'transformer.blocks.0.geom_attn.s_norm', 'transformer.blocks.0.geom_attn.proj', 'transformer.blocks.0.geom_attn.out_proj', 'transformer.blocks.0.geom_attn', 'transformer.blocks.0.ffn.0', 'transformer.blocks.0.ffn.1', 'transformer.blocks.0.ffn.2', '

In [31]:
with torch.no_grad():
    output2,cache = esm3_hooked1.run_with_cache(
        sequence_tokens=tokens
    )

In [33]:
cache.keys()

dict_keys(['hook_embed', 'blocks.0.hook_resid_pre', 'blocks.0.hook_post_layer_norm', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_ln_q', 'blocks.0.attn.hook_ln_k', 'blocks.0.attn.hook_rot_q', 'blocks.0.attn.hook_rot_k', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.hook_geo_attn_in', 'blocks.0.hook_geo_attn_out', 'blocks.0.hook_resid_mid_geo', 'blocks.0.hook_mlp_in', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.hook_post_layer_norm', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_ln_q', 'blocks.1.attn.hook_ln_k', 'blocks.1.attn.hook_rot_q', 'blocks.1.attn.hook_rot_k', 'blocks.1.attn.hook_z', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.hook_mlp_in', 'blocks.1.mlp.hook_pre', 'blocks.1.mlp.hook_post', 'blocks.1.hook_mlp_out', 'blocks.1.hook_resi

In [34]:
assert torch.equal (cache['hook_embed'], activations_outputs['encoder'])
assert torch.equal (cache['blocks.0.hook_resid_pre'], activations_inputs['transformer.blocks.0.attn.layernorm_qkv.0'][0])
#assert torch.equal (cache['blocks.0.hook_q_input'][:, :, 0,:], cache['blocks.0.hook_q_input'][:, :, 23,:])
assert torch.equal (cache['blocks.0.hook_post_layer_norm'][:, :, 23,:], cache['blocks.0.hook_post_layer_norm'][:, :, 15,:])
assert torch.equal (cache['blocks.0.hook_post_layer_norm'][:, :, 23,:], activations_outputs['transformer.blocks.0.attn.layernorm_qkv.0'])


IndexError: too many indices for tensor of dimension 3

In [ ]:
activations_inputs['transformer.blocks.0.attn.q_ln'][0][:,:,0:63]

In [ ]:
cache['blocks.0.attn.hook_q'][:,:,0,:]

In [25]:
torch.equal(activations_inputs['transformer.blocks.0.attn.q_ln'][0][:,:,0:64], cache['blocks.0.attn.hook_q'][:,:,0,:])

False

In [ ]:
torch.max(torch.abs(activations_inputs['transformer.blocks.0.attn.q_ln'][0][:,:,0:64]-cache['blocks.0.attn.hook_q'][:,:,0,:]))

In [35]:
activations_outputs['transformer.blocks.0.attn.layernorm_qkv.0']

tensor([[[ 0.0602, -0.0695,  0.0404,  ...,  0.0348, -0.0713,  0.0166],
         [-0.0555,  0.0565,  0.0367,  ...,  0.0177,  0.0813,  0.0349],
         [ 0.0412, -0.0076,  0.0131,  ...,  0.0267,  0.0452, -0.0143],
         ...,
         [-0.0155,  0.0216, -0.0443,  ...,  0.0107,  0.0081, -0.0536],
         [ 0.0412, -0.0076,  0.0131,  ...,  0.0267,  0.0452, -0.0143],
         [ 0.0243, -0.0021,  0.0028,  ...,  0.0458, -0.0569,  0.0081]]],
       device='cuda:0')

In [36]:
def complex_attn_linear(
    input: Float[torch.Tensor, "batch pos head_index d_model"],
    w: Float[torch.Tensor, "head_index d_model d_head"],
    b: Float[torch.Tensor, "head_index d_head"],
) -> Float[torch.Tensor, "batch pos head_index d_head"]:
    """Linear layer for attention calculation.

    This is almost the same as simple_attn_linear, but the input tensor has an extra head_index dimension, used when calculating the input of each attention head separately.
    """

    # Add singleton dimensions for broadcasting
    input = einops.rearrange(
        input, "batch pos head_index d_model -> batch pos head_index d_model 1"
    )
    w = einops.rearrange(w, "head_index d_model d_head -> 1 1 head_index d_model d_head")

    # Element-wise multiplication and sum over the d_model dimension
    result = input * w
    result = result.sum(dim=-2)
    print("hi")
    return result
def simple_attn_linear(
    input: Float[torch.Tensor, "batch pos d_model"],
    w: Float[torch.Tensor, "head_index d_model d_head"],
    b: Float[torch.Tensor, "head_index d_head"],
    W_org,
    b_org
) -> Float[torch.Tensor, "batch pos head_index d_head"]:
    """Linear layer for attention calculation."""
    w = einops.rearrange(w, "head_index d_model d_head -> (head_index d_head) d_model")
    b_ = einops.rearrange(b, "head_index d_head -> (head_index d_head)")
    if torch.all(b_== 0):
        b_=None
    assert torch.equal(W_org,w)
    assert b_ is None
    assert b_org is None
    return F.linear(input, w, b_)

In [37]:
#input to linear:
from transformer_lens.utils import repeat_along_head_dimension

input_to_linear = repeat_along_head_dimension(activations_outputs['transformer.blocks.0.attn.layernorm_qkv.0'], n_heads=esm3_hooked1.cfg.n_heads)
w = esm3_hooked1.blocks[0].attn.W_Q
b = esm3_hooked1.blocks[0].attn.b_Q
res=complex_attn_linear(input=input_to_linear, w=w, b=b)

hi


In [38]:
in2 = activations_outputs['transformer.blocks.0.attn.layernorm_qkv.0']
W_org = esm3_original1.transformer.blocks[0].attn.layernorm_qkv[1].weight
b_org = esm3_original1.transformer.blocks[0].attn.layernorm_qkv[1].bias
res2= simple_attn_linear(in2,w,b,W_org, b_org )

AssertionError: 

In [39]:
torch.equal(activations_inputs['transformer.blocks.0.attn.q_ln'][0], res2)

NameError: name 'res2' is not defined

In [40]:
W_org = esm3_original1.transformer.blocks[0].attn.layernorm_qkv[1].weight
b_org = esm3_original1.transformer.blocks[0].attn.layernorm_qkv[1].bias
res_org = F.linear(in2, (W_org[0:1536]), b_org)

In [41]:
torch.equal(res_org, activations_inputs['transformer.blocks.0.attn.q_ln'][0])

False

In [42]:
W_org[0:1536].shape

torch.Size([1536, 1536])

In [43]:
out_linear_block_0 = activations_outputs['transformer.blocks.0.attn.layernorm_qkv.1']

In [44]:
out_linear_block_0.shape

torch.Size([1, 105, 4608])

In [45]:
torch.equal(out_linear_block_0[:,:,0:1536], res_org)

False

In [46]:
def test1(device):
    w = torch.randn(size=(1536, 1536*3)).to(device)
    x = torch.randn(size=(3,10,1536)).to(device)

    a= x@w
    b= x @ (w[:, 0:1536])
    assert torch.equal(a[:, :, 0:1536], b)  

In [47]:
test1("cuda")

AssertionError: 